In [8]:
import os
from pathlib import Path
from ures.files import filter_files
from transformers import AutoModelForCausalLM, DataCollatorForLanguageModeling, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM, AutoModelForMaskedLM
from perf_estimator.config import Config

In [9]:
target_dir = Path("/home/glaswegian/Documents/301-ResearchData/101-xMem-LLM/004-Analysis Custom Training Loop")
target_dir = Path("/home/glaswegian/DL-Estimator")
# target_dir = Path("/home/glaswegian/.cache/XMemEstimator")
# target_dir = Path("/home/mechrevo/Documents/101-ResearchData/101-xMem-LLm/004-Analysis Custom Training Loop")
p_files = filter_files("pt.trace.json", target_dir, fuzz=True)
p_files

['/home/glaswegian/DL-Estimator/20250415-195532-662d/results/profiler/glaswegian-Z890-GAMING-X-WIFI7_5854.1744747837535075284.pt.trace.json',
 '/home/glaswegian/DL-Estimator/20250415-195532-662d/results/profiler/glaswegian-Z890-GAMING-X-WIFI7_5854.1744747974196142361.pt.trace.json']

In [10]:
n_files = filter_files("host_metrics", target_dir, fuzz=True)
n_files


[]

In [11]:

# 1. 加载模型和分词器
model_name = "EleutherAI/gpt-neo-125M"
model_name = "facebook/opt-125m"
# model_name = "cerebras/Cerebras-GPT-111M"
# model_name = "bigscience/bloom-560m" # I should try it on CoLab
# model_name = "mosaicml/mpt-7b"
model = AutoModelForCausalLM.from_pretrained(model_name)

# model_name = "t5-base"
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


# model_name = "microsoft/deberta-base"
# model = AutoModelForMaskedLM.from_pretrained(model_name)



In [12]:

from perf_estimator.estimator import TrainerEstimator
from perf_estimator.dataset import image_dataset
file_index = -1
config = Config()
config.trainer.huggingface_enable = True
config.trainer.huggingface_model_name = model_name
estimator = TrainerEstimator(
    dataloader=image_dataset(batch=100),
    profiler_file=p_files[file_index],
    max_gpu_memory_in_gb=7.6,
    config=config,
)

# Model Memory

In [13]:
e_model_memory = estimator.model_memory(iteration_index=1, reset=True)

In [14]:
memory_blocks = []
global_index = 0
for index, (name, params) in enumerate(model.named_parameters()):
    global_index += 1
    para_size = params.nelement() * params.element_size()
    block = estimator.create_memory_block(
        byte=para_size,
        timestamp=global_index
    )
    memory_blocks.append(block)

for index, buffer in enumerate(model.buffers()):
    global_index += 1
    buffer_size = buffer.nelement() * buffer.element_size()
    block = estimator.create_memory_block(
        byte=buffer_size,
        timestamp=global_index
    )
    memory_blocks.append(block)


In [15]:
model_allocator, model_result = estimator.estimate_memory_blocks(memory_blocks)
model_allocator.plot_memory_change()

e_model_allocator, model_result = estimator.estimate_memory_blocks(e_model_memory)
e_model_allocator.plot_memory_change()



# Training Estimation

In [16]:
allocator, result = estimator.estimate()
print(f"Estimate memory: {max(allocator._trace.max_segment_changes)/1024**3}")
allocator.plot_memory_change()

Estimate memory: 3.67578125


In [17]:
import json
def get_truth_ground_max_gpu(n_file):
    with open(n_file, "r") as f:
        _data = json.load(f)
    _gpu_memory_usage = {}
    start_memory = {}
    for index, gpu_metric in enumerate(_data["records"]):
        gpu_data = gpu_metric["HostGPUs"]
        for device_id, data in gpu_data.items():
            if index == 0:
                start_memory[device_id] = data["memory"]["used"]
            if device_id not in _gpu_memory_usage:
                _gpu_memory_usage[device_id] = []
            _gpu_memory_usage[device_id].append(
                data["memory"]["used"] - start_memory[device_id]
            )
    return _gpu_memory_usage

max(get_truth_ground_max_gpu(n_files[-8])['0'])

IndexError: list index out of range